# Tutorial 02: Dynamic Web Scraping
Author: Maximilian Kreutner

In this notebook, we will learn how to scrape **Dynamic Websites** using [**Selenium**](https://www.selenium.dev/).

Last week we looked at `requests` and `BeautifulSoup`. These tools work by downloading the **HTML Source Code** of a page. This works great for static sites (like Wikipedia), but fails on modern web apps (like X (Twitter) or LinkedIn) where data is loaded via **JavaScript**. We will use the site **[http://quotes.toscrape.com/js/](http://quotes.toscrape.com/js/)** as an example.


In [1]:
# Install necessary libraries into your conda/uv environment
# !pip install selenium webdriver-manager pandas requests beautifulsoup4

When you use Python's `requests`, you get the **Source Code** (the raw file from the server *before* JavaScript runs).
This fails for dynamic websites:

In [ ]:
import requests
from bs4 import BeautifulSoup

# This is a static webpage
static_url = "http://quotes.toscrape.com/"

# This is a dynamic webpage, which uses javascript to load the data
dynamic_url = "http://quotes.toscrape.com/js/"

# Test both responses to see if they work
# response = requests.get(static_url)
response = requests.get(dynamic_url)

soup = BeautifulSoup(response.text, 'html.parser')

# We search for the quote element
quote = soup.find(class_="quote")

print(f"Status Code: {response.status_code}") # 200 means the page loaded successfully

if quote:
    print(quote.text)
else:
    print("ERROR: We found no quotes.")
    print("Reason: The HTML is empty because JavaScript hasn't run yet.")

Status Code: 200
ERROR: We found no quotes.
Reason: The HTML is empty because JavaScript hasn't run yet.


## Setting up Selenium
Since the data is generated by JavaScript, we need a tool that can simulate a "real" browser.

**Selenium** opens a real instance of Chrome, waits for the JavaScript to execute, and then allows us to extract the data.


We import additional helpful classes.

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Selenium also works for firefox, but commands and later options are slightly different.
# For this tutorial we will stick to chrome.
# from webdriver_manager.firefox import GeckoDriverManager
# options = webdriver.FirefoxOptions()

options = webdriver.ChromeOptions()

# No display of browser -> faster. For demonstration we will keep the default mode
# options.add_argument("--headless=new") 

# ChromeDriverManager downloads the correct driver for your Chrome version automatically
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

Load the first page with `driver.get(dynamic_url)`. We need to wait for the first element to appear.

In [ ]:
# load the page
driver.get(dynamic_url)

# We can use time.sleep(n) to wait for the page to appear, but WebDriverWait can be more efficient.
wait = WebDriverWait(driver, 10)

try:
    # Wait max 10 seconds until class="quote" is present
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))
    print("JavaScript has rendered. Content is ready!")
except Exception as e:
    print(f"Timeout: {e}")

JavaScript has rendered. Content is ready!


## Scraping the page
Now that the page is loaded, we can define a function to extract the data.

We will need the following two methods:
1. `driver.find_elements` to get the list of quote cards.
2. `card.find_element` to get specific details inside each card.

Implement the method `scrape_current_page` to extract the quoutes, authors and tags from the current visible page. It should return a list, where each item in the list contains a dictionary with `text`, `author` and `tags` as strings. The tags can be concatenated with `, `.

In [5]:
def scrape_current_page(driver):
    """Extracts quotes, authors and tags from the currently visible page."""
    
    page_data = []
    
    quote_cards = driver.find_elements(By.CLASS_NAME, "quote")
    
    for card in quote_cards:
        try:
            text = card.find_element(By.CLASS_NAME, "text").text
            
            author = card.find_element(By.CLASS_NAME, "author").text
            
            tag_elements = card.find_elements(By.CLASS_NAME, "tag")
            tags = [t.text for t in tag_elements]
            
            item = {
                "text": text,
                "author": author,
                "tags": ", ".join(tags)
            }
            page_data.append(item)
        except Exception as e:
            print(f"Error parsing card: {e}")
            
    return page_data

# Test the function on the first page and the first quote
first_page_data = scrape_current_page(driver)
print(f"Extracted {len(first_page_data)} quotes from the first page.")
print(f"TEXT: {first_page_data[0]["text"]}")
print(f"AUTHOR: {first_page_data[0]["author"]}")
print(f"TAGS: {first_page_data[0]["tags"]}")


Extracted 10 quotes from the first page.
TEXT: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
AUTHOR: Albert Einstein
TAGS: change, deep-thoughts, thinking, world



## Pagination
Some quotes are on different pages. We need to click a button to make them appear.

We will:
1. Scrape the current page.
2. Find the "Next" button.
3. Click it.
4. Wait for the new quotes to load.
5. Repeat.


Use the method from before to scrape the quotes on the current page. Then use `next_btn = driver.find_element()` to find the next button and use the `next_btn.click()` to switch to the next page.

In [ ]:
all_quotes = []
pages_to_scrape = 3 # Limiting to 3 pages for this example

# We are already on Page 1, so we start the loop
for page_num in range(pages_to_scrape):
    print(f"Scraping Page {page_num + 1}...")
    
    # Use our scrape function from earlier
    data = scrape_current_page(driver)
    all_quotes.extend(data)
    
    # Now we need to click the next button
    try:
        # Look for the 'Next' button
        next_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "Next")
        
        # CSS Selector can be more precise:
        # CSS Selector 'li.next > a' means: An <a> tag inside an <li> with class 'next'
        # next_btn = driver.find_element(By.CSS_SELECTOR, "li.next > a")

        next_btn.click()
        
        # Wait until the next page to load. You can use a simple sleep, but to be more efficient use wait again.
        # time.sleep(2) 
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))
        
        
    except Exception as e:
        print("No more pages found (or last page reached).")
        break

print(f"Done! Collected {len(all_quotes)} total quotes.")

Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Done! Collected 30 total quotes.


Once we are done, we close down the driver.

In [7]:
driver.quit()

## Use the data

Convert the your data into a pandas `DataFrame` and print out all quotes that contain the tag `inspirational`.

In [8]:
df = pd.DataFrame(all_quotes)

inspirational_quotes = df[df['tags'].str.contains("inspirational")]

for index, row in inspirational_quotes.iterrows():
    print(f"Author: {row['author']}")
    print(f"Quote:  {row['text']}")
    print("-" * 50)

Author: Albert Einstein
Quote:  “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
--------------------------------------------------
Author: Marilyn Monroe
Quote:  “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
--------------------------------------------------
Author: Thomas A. Edison
Quote:  “I have not failed. I've just found 10,000 ways that won't work.”
--------------------------------------------------
Author: Marilyn Monroe
Quote:  “This life is what you make it. No matter what, you're going to mess up sometimes, it's a universal truth. But the good part is you get to decide how you're going to mess it up. Girls will be your friends - they'll act like it anyway. But just remember, some come, some go. The ones that stay with you through everything - they're your true best friends. Don't let go of them. Also remember, sisters make the b